# Estimating the material content of the global vehicle fleet - step 1: inflow-driven model for the use phase
*derived from flodym example 5*

This example shows an inflow-driven model to estimate the passenger vehicle fleet in 2024, covering multiple countries and the years 1993-2024.

The research questions asked is: How big is the passenger vehicle fleet in the different countries based on the inflow-driven model? Is the result realistic/correct, and how does a changing vehicle lifetime impact the resulting fleet size?

To answer these questions a dynamic material flow analysis is performed.

The dynamic MFA model has the following indices:

* t: time (1993-2024)
* r: region (39 countries for which inflow data are easily available)
* g: good (passenger vehicle) (only for the second part)

The system definition of the model is given in the figure below. **This introductry exercise focusses on the use phase only.**

<img src="pictures/System_example5.png" width="850" height="290" alt="MFA system">

The model equations are as follows:

1) inflow-driven dynamic stock model, where $F_{1-2}$ is the historic inflow, $Sf$ is the survival function of the age-cohort (1-sum(pdf of discard)), and $S_2$ is the stock:
$S_2(t,c,r,g) = F_{1-2}(c,r,g)\cdot Sf(t,c,r,g)$


## 1. Load flodym and other useful packages

In [ ]:
from os.path import join

import numpy as np
import pandas as pd
import plotly.express as px

from flodym import (
    DimensionDefinition,
    Dimension,
    DimensionSet,
    ParameterDefinition,
    Parameter,
    FlowDefinition,
    StockDefinition,
    MFASystem,
    MFADefinition,
    DataReader,
    InflowDrivenDSM,
)
from flodym.lifetime_models import NormalLifetime

## 2. Define the data requirements, flows, stocks and MFA system equations

We define the dimensions that are relevant for our system and the model parameters, processes, stocks and flows. We further define a class with our system equations in the compute method.

In [ ]:
dimension_definitions = [
    DimensionDefinition(letter="t", name="time", dtype=int),
    DimensionDefinition(letter="r", name="region", dtype=str),
]

process_names = ["sysenv", "use"]

parameter_definitions = [
    ParameterDefinition(name="vehicle lifetime", dim_letters=("r",)),
    ParameterDefinition(name="vehicle new registration", dim_letters=("t", "r")),
]

flow_definitions = [
    FlowDefinition(from_process_name="sysenv", to_process_name="use", dim_letters=("t", "r")),
    FlowDefinition(from_process_name="use", to_process_name="sysenv", dim_letters=("t", "r")),
]

stock_definitions = [
    StockDefinition(
        name="in use",
        process="use",
        dim_letters=("t", "r"),
        subclass=InflowDrivenDSM,
        lifetime_model_class=NormalLifetime,
        time_letter="t",
    )
]

mfa_definition = MFADefinition(
    dimensions=dimension_definitions,
    processes=process_names,
    flows=flow_definitions,
    stocks=stock_definitions,
    parameters=parameter_definitions,
)

In [ ]:
class VehicleMFA(MFASystem):
    """We just need to define the compute method with our system equations,
    as all the other things we need are inherited from the MFASystem class."""

    def compute(self):
        self.compute_stock()
        self.compute_flows()

    def compute_stock(self):
        self.stocks["in use"].inflow[...] = self.parameters["vehicle new registration"]
        self.stocks["in use"].lifetime_model.set_prms(
            mean=self.parameters["vehicle lifetime"],
            std=self.parameters["vehicle lifetime"] * 0.3,
        )
        self.stocks["in use"].compute()

    def compute_flows(self):
        self.flows["sysenv => use"][...] = self.parameters["vehicle new registration"]
        self.flows["use => sysenv"][...] = self.stocks["in use"].outflow


## 3. Define our data reader
Now that we have defined the MFA system and know what data we need, we can load the data.
To do the data loading, we define a DataReader class. Such a class can be reused with different datasets of the same format by passing attributes, e.g. the directory where the data is stored, in the init function. In this example, we will also use this data reader in a following step.

In [ ]:
class CustomDataReader(DataReader):
    """The methods `read_dimensions` and `read_parameters` are already defined in the parent
    DataReader class, and loop over the methods `read_dimension` and `read_parameter_values`
    that we specify for our usecase here.
    """

    def __init__(self, data_directory, years):
        self.data_directory = data_directory
        self.years = years

    def read_dimension(self, dimension_definition: DimensionDefinition) -> Dimension:
        if (dim_name := dimension_definition.name) == "region":
            data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_new_registration.xlsx"), "Data"
            )
            other_data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime.xlsx"), "Data"
            )
            data = (set(data[dim_name].unique())).intersection(set(other_data[dim_name].unique()))
            data = list(data)
            data.sort()
        elif dimension_definition.name == "time":
            data = self.years
        return Dimension(
            name=dimension_definition.name,
            letter=dimension_definition.letter,
            dtype=dimension_definition.dtype,
            items=data,
        )

    def read_parameter_values(self, parameter_name: str, dims: DimensionSet) -> Parameter:
        data = pd.read_excel(
            join(self.data_directory, f"flodym_Bath_dMFA_Global_Vehicle_Fleet_{parameter_name.replace(' ', '_')}.xlsx"), "Data"
        )
        data = data.fillna(0)
        if "r" in dims:  # remove unwanted regions
            data = data[data["region"].isin(dims["r"].items)]

        if parameter_name == "vehicle lifetime":
            data.columns = [x.lower() for x in data.columns]
            # remove unncessary columns
            data = data[[dim.name for dim in dims] + ["value"]]
        return Parameter.from_df(dims=dims, df=data)

In [ ]:
'''
# Test code
data = pd.read_excel(
    join("input_data", "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_new_registration.xlsx"), "Data"
)
other_data = pd.read_excel(
    join("input_data", "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime.xlsx"), "Data"
)
data = (set(data["region"].unique())).intersection(set(other_data["region"].unique()))
data = list(data)
data.sort()
len(data)
data
'''

## 4. Put everything together
We make an instance of our `CustomDataReader`, read in the data and use it to create an instance of our `VehicleMFA` class. Then we can run the calculations, and check what our estimate of vehicle stocks looks like compared to the data for 2015 in the `vehicle_stock.xlsx` file.

In [ ]:
data_reader = CustomDataReader(data_directory="input_data", years=list(range(1993, 2025)))

vehicle_mfa = VehicleMFA.from_data_reader(
    definition=mfa_definition,
    data_reader=data_reader,
)
vehicle_mfa.compute()  

## 5. Answer research question
How big is the passenger vehicle fleet in the different countries based on the inflow-driven model?

In [ ]:
vehicle_mfa.stocks["in use"].stock
stock_df = vehicle_mfa.stocks["in use"].stock.to_df(index=False) # results in list-shaped dataframe
stock_df
stock_df = vehicle_mfa.stocks["in use"].stock.to_df(index="time", dim_to_columns="region") # results in table-shaped dataframe
stock_df

In [ ]:
fig = px.line(stock_df, title="passenger vehicle fleet, build-up since 1993")
fig.show(renderer="notebook")


Note that this stock plot only shows the age-cohorts since 1993, older age-cohorts are not present in the dataset, as no inflow is reported for these years.
The stock shown here thus does not contain age-cohort older than 1993, which means that after 2020, the estimates are reasonable accurate for most countries (unlike the the actual average lifetime is much longer than 16 years).

**Note** that this inflow-driven model applies a fixed lifetime to all age-cohorts, the resulting stock is therefore just an estimate and may differ substantially from the actually reported fleet size and composition. The lifetime can be adjusted to calibrate the stock estimate to the reported numbers.

You can change the lifetime data, either the values in the xlsx, the standard deviation (0.3*mean) in the code above, or scale the whole lifetime (1.1*...) in the code above.

## 6. Add a new dimension - two different drive technologies at the same time
Now, introduce a new dimension to the system
    "DimensionDefinition(letter="T", name="drive technology", dtype=str),"
to calculate two drive technologies at the same time.
Use the parameters flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_new_registration_by_technology and flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime_by_technology instead

In [ ]:
dimension_definitions = [
    DimensionDefinition(letter="t", name="time", dtype=int),
    DimensionDefinition(letter="r", name="region", dtype=str),
    DimensionDefinition(letter="T", name="drive technology", dtype=str),
]

# continue adding code here...

In [ ]:
class VehicleMFA(MFASystem):
    """We just need to define the compute method with our system equations,
    as all the other things we need are inherited from the MFASystem class."""

# continue adding code here...


In [ ]:
class CustomDataReader(DataReader):
    """The methods `read_dimensions` and `read_parameters` are already defined in the parent
    DataReader class, and loop over the methods `read_dimension` and `read_parameter_values`
    that we specify for our usecase here.
    """

    def __init__(self, data_directory, years):
        self.data_directory = data_directory
        self.years = years

    def read_dimension(self, dimension_definition: DimensionDefinition) -> Dimension:
        if (dim_name := dimension_definition.name) == "region":
            data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_new_registration_by_technology.xlsx"), "Data"
            )
            other_data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime_by_technology.xlsx"), "Data"
            )
            data = (set(data[dim_name].unique())).intersection(set(other_data[dim_name].unique()))
            data = list(data)
            data.sort()
        elif dimension_definition.name == "drive technology":
            # continue adding code here...   
        elif dimension_definition.name == "time":
            data = self.years
        return Dimension(
            name=dimension_definition.name,
            letter=dimension_definition.letter,
            dtype=dimension_definition.dtype,
            items=data,
        )

    def read_parameter_values(self, parameter_name: str, dims: DimensionSet) -> Parameter:
        data = pd.read_excel(
            join(self.data_directory, f"flodym_Bath_dMFA_Global_Vehicle_Fleet_{parameter_name.replace(' ', '_')}.xlsx"), "Data"
        )
        data = data.fillna(0)
        if "r" in dims:  # remove unwanted regions
            data = data[data["region"].isin(dims["r"].items)]

        if parameter_name == "vehicle lifetime by technology":
            data.columns = [x.lower() for x in data.columns]
            # remove unncessary columns
            data = data[[dim.name for dim in dims] + ["value"]]
        return Parameter.from_df(dims=dims, df=data)

In [ ]:
data_reader = CustomDataReader(data_directory="input_data", years=list(range(1993, 2025)))

vehicle_mfa = VehicleMFA.from_data_reader(
    definition=mfa_definition,
    data_reader=data_reader,
)
vehicle_mfa.compute()  

In [ ]:
# Inspect results. continue adding code here...
vehicle_mfa.stocks["in use"].stock
stock_df = vehicle_mfa.stocks["in use"].stock.to_df(index=False) # results in list-shaped dataframe
stock_df

Plot the stocks for the two drive technologies and for selected regions

In [ ]:
stock_df_filter = stock_df[(stock_df['drive technology'] == 'electric passenger cars')].drop(columns=['drive technology'])
#stock_df_filter_plot = stock_df_filter.pivot(index='time', columns='region') # leads to multi-index, which cannot be plotted by plotly express
stock_df_filter_plot = stock_df_filter.set_index(['time','region']).squeeze().unstack('region')
stock_df_filter_plot

In [ ]:
fig = px.line(stock_df_filter_plot, title="passenger vehicle fleet, electric passenger cars, build-up since 1993")
fig.show(renderer="notebook")

## 7. Inspect stock by age-cohort

In [ ]:
vehicle_mfa.stocks["in use"].get_stock_by_cohort().shape

In [ ]:
fig = px.imshow(vehicle_mfa.stocks["in use"].get_stock_by_cohort()[:,:,1,1]) # chose a region (index position 3 and technology (pos. 4)
fig.show()


## 8 export stock time series for later use

In [ ]:
vehicle_mfa.stocks["in use"].stock
stock_df = vehicle_mfa.stocks["in use"].stock.to_df(index=False) # results in list-shaped dataframe
stock_df.set_index(['time','drive technology','region']).unstack('time').to_excel("results/RESULT_Inflow_Driven_Stock.xlsx", merge_cells = False)